In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Install Dependencies & Setup Environment

In [2]:
import os
import sys
import glob
import json
import re
import copy
import warnings
import numpy as np
import pandas as pd
import cv2
from PIL import Image, ImageOps
from shutil import copyfile
from sklearn.impute import KNNImputer
from sklearn.model_selection import GroupShuffleSplit, StratifiedKFold

warnings.filterwarnings('ignore')

# Config Path and Constant

In [3]:
BASE_INPUT = '/kaggle/input/datasets/akhyaruu/appendicitis'
BASE_OUTPUT = '/kaggle/working/Processed'

# Directory setup
IMAGE_DIR = os.path.join(BASE_INPUT, 'US_Pictures_kaggle_ready')
TRAIN_IMAGE_DIR = os.path.join(BASE_OUTPUT, 'Train')
TEST_IMAGE_DIR = os.path.join(BASE_OUTPUT, 'Test')
PREPROC_IMAGE_DIR = os.path.join(BASE_OUTPUT, 'Preprocessed')
OUTPUT_DIR = os.path.join(BASE_OUTPUT, 'Dictionaries')

CLINICAL_DATA_FILE = os.path.join(BASE_INPUT, 'app_data.xlsx')

# Blacklist dummy (jika tidak ada)
BLACKLIST_FILE = os.path.join(BASE_INPUT, 'blacklist.csv')
if not os.path.exists(BLACKLIST_FILE):
    BLACKLIST_FILE = '/kaggle/working/empty_blacklist.csv'
    with open(BLACKLIST_FILE, 'w') as f:
        f.write("image_name\n")

# Konfigurasi
PADDING_MODE = 'constant'
TARGET_LABEL = 'diagnosis'

# Sementara pakai fitur numerical
# TABULAR_FEATURES = ['Age', 'Sex', 'Height', 'Weight', 'BMI', 'Alvarado_Score', 'Paedriatic_Appendicitis_Score',
#                     'Peritonitis', 'Migratory_Pain', 'Lower_Right_Abd_Pain', 'Contralateral_Rebound_Tenderness', 
#                     'Coughing_Pain', 'Psoas_Sign', 'Nausea', 'Loss_of_Appetite', 'Body_Temperature', 'Dysuria', 
#                     'Stool', 'WBC_Count', 'Neutrophil_Percentage', 'CRP', 'Ketones_in_Urine', 'RBC_in_Urine', 
#                     'WBC_in_Urine', 'Appendix_on_US', 'Appendix_Diameter', 'Free_Fluids', 
#                     'Appendix_Wall_Layers', 'Target_Sign', 'Perfusion', 'Perforation', 
#                     'Surrounding_Tissue_Reaction', 'Pathological_Lymph_Nodes', 'Bowel_Wall_Thickening', 'Ileus', 
#                     'Coprostasis', 'Meteorism', 'Enteritis', 'Appendicular_Abscess', 
#                     'Conglomerate_of_Bowel_Loops', 'Gynecological_Findings']

TABULAR_FEATURES = ['Age', 'BMI', 'Height', 'Weight', 'Alvarado_Score', 'Paedriatic_Appendicitis_Score']

REAL_VALUED = ['Age', 'Height', 'Weight', 'BMI', 'Body_Temperature', 'WBC_Count', 'Neutrophil_Percentage', 
               'CRP', 'Appendix_Diameter']

CONCEPTS = ['Appendix_on_US', 'Free_Fluids', 'Appendix_Wall_Layers', 'Target_Sign', 'Surrounding_Tissue_Reaction',
            'Pathological_Lymph_Nodes', 'Bowel_Wall_Thickening', 'Coprostasis', 'Meteorism', 'Enteritis', 
            'Appendix_Diameter', 'Perforation', 'Appendicular_Abscess', 'Conglomerate_of_Bowel_Loops', 
            'Gynecological_Findings']

# Buat direktori output
for d in [TRAIN_IMAGE_DIR, TEST_IMAGE_DIR, PREPROC_IMAGE_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

print("Konfigurasi Path Selesai.")

Konfigurasi Path Selesai.


# Split Data & Rename File

In [4]:
def test_split(train_image_dir, test_image_dir):
    train_files = glob.glob(os.path.join(train_image_dir, '*'))
    test_files = glob.glob(os.path.join(test_image_dir, '*'))
    
    train_groups = np.array([int(re.split('_| |\\.', os.path.basename(f))[0]) for f in train_files])
    test_groups = np.array([int(re.split('_| |\\.', os.path.basename(f))[0]) for f in test_files])
    
    print(f"Train: {len(np.unique(train_groups))} patients, {len(train_groups)} images")
    print(f"Test: {len(np.unique(test_groups))} patients, {len(test_groups)} images")
    
    if len(np.intersect1d(train_groups, test_groups)) == 0:
        print("✅ Patients in train and test set don't overlap!")

# 1. Split dataset
print("Memisahkan Train dan Test...")
image_file_list = glob.glob(os.path.join(IMAGE_DIR, '*'))
image_names = [os.path.basename(f) for f in image_file_list]
groups = np.array([int(re.split('_| |\\.', name)[0]) for name in image_names])

gss = GroupShuffleSplit(n_splits=1, train_size=.9, random_state=42)
for idx1, idx2 in gss.split(image_names, groups=groups):
    train_idx, test_idx = idx1, idx2

for image_idx, image_name in enumerate(image_names):
    src_path = os.path.join(IMAGE_DIR, image_name)
    if image_idx in train_idx:
        copyfile(src_path, os.path.join(TRAIN_IMAGE_DIR, image_name))
    else:
        copyfile(src_path, os.path.join(TEST_IMAGE_DIR, image_name))

test_split(TRAIN_IMAGE_DIR, TEST_IMAGE_DIR)

# 2. Rename file aneh (spasi, umlaut, dll)
def rename_weird_files(folder):
   files = glob.glob(folder + '/*')
   for f in files:
      name = os.path.basename(f)
      if any(c in name for c in ['ä', 'Ä', 'ö', 'Ö', 'ü', 'Ü', 'ß', ' ']):
         try:
            img = Image.open(f)
            new_name = name.replace(' ', '_').replace('ä', 'ae').replace('Ä', 'Ae').replace('ö', 'oe').replace('Ö', 'Oe').replace('ü', 'ue').replace('Ü', 'Ue').replace('ß', 'ss')
            new_path = os.path.join(folder, new_name)
            img.save(new_path)
            os.remove(f)
         except Exception:
            pass

print("Membersihkan nama file...")
rename_weird_files(TRAIN_IMAGE_DIR)
rename_weird_files(TEST_IMAGE_DIR)

Memisahkan Train dan Test...
Train: 636 patients, 1870 images
Test: 71 patients, 227 images
✅ Patients in train and test set don't overlap!
Membersihkan nama file...


# Pembersihan Gambar (DeepFill & Cropping)

In [5]:
def process_images(src_dir, out_dir, target_size=(224, 224)):
    print(f"Memproses (Inpainting + Crop) gambar dari {os.path.basename(src_dir)}...")
    os.makedirs(out_dir, exist_ok=True)
    files = glob.glob(os.path.join(src_dir, '*'))
    
    for f in files:
        filename = os.path.basename(f)
        img = cv2.imread(f)
        if img is None: continue
            
        # 1. Inpainting (DeepFill Replacement)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        _, mask = cv2.threshold(gray, 240, 255, cv2.THRESH_BINARY) # Deteksi teks putih
        img_inpainted = cv2.inpaint(img, mask, 3, cv2.INPAINT_TELEA)
        
        # 2. Cropping & Padding
        img_rgb = cv2.cvtColor(img_inpainted, cv2.COLOR_BGR2RGB)
        # img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(img_rgb)
        img_padded = ImageOps.pad(pil_img, target_size, color=(0, 0, 0))
        
        # Simpan
        img_padded.save(os.path.join(out_dir, filename))

# Jalankan proses
TRAIN_PROCESSED = os.path.join(PREPROC_IMAGE_DIR, 'cropped_train')
TEST_PROCESSED = os.path.join(PREPROC_IMAGE_DIR, 'cropped_test')

process_images(TRAIN_IMAGE_DIR, TRAIN_PROCESSED)
process_images(TEST_IMAGE_DIR, TEST_PROCESSED)

Memproses (Inpainting + Crop) gambar dari Train...
Memproses (Inpainting + Crop) gambar dari Test...


# Generate JSON & Imputasi

In [6]:
def generate_json_mapping_dropna(image_dir, excel_file, output_filepath, target_label='Diagnosis'):
    print(f"Membuat mapping JSON untuk {os.path.basename(image_dir)} (Mode: Drop NaN)...")
    df = pd.read_excel(excel_file)
    
    # 1. Bersihkan nama kolom dari spasi tak terlihat
    df.columns = df.columns.str.strip()
    
    # 2. Buat ID Pasien berdasarkan urutan baris (Index + 1)
    df['Patient_ID'] = df.index + 1
    
    # 3. Ambil hanya kolom yang kita butuhkan (Fitur + Konsep + Label + ID)
    # Jika ada kolom yang tidak ada di Excel, ini akan error, jadi pastikan namanya persis.
    cols_to_keep = ['Patient_ID', target_label] + TABULAR_FEATURES + CONCEPTS
    # Kita filter hanya kolom yang memang ada di DataFrame (untuk menghindari KeyError)
    cols_to_keep = [c for c in cols_to_keep if c in df.columns]
    df_clean = df[cols_to_keep].copy()
    
    # 4. DROP BARIS YANG MEMILIKI NILAI KOSONG (NaN) DI FITUR ATAU LABEL
    # Kita hanya drop jika fitur atau label yang kosong. Konsep yang kosong juga di-drop.
    print(f"Jumlah data awal di Excel: {len(df_clean)} baris")
    df_clean = df_clean.dropna(subset=TABULAR_FEATURES + [target_label]) 
    print(f"Jumlah data setelah drop NaN: {len(df_clean)} baris")
    
    available_images = glob.glob(os.path.join(image_dir, '*'))
    mapping = {}
    
    for img_path in available_images:
        filename = os.path.basename(img_path)
        try:
            # Ambil ID dari nama file (contoh: "101.1_App.bmp" -> 101)
            patient_id = int(filename.split('.')[0].strip())
            
            # Cari baris dengan ID tersebut
            row = df_clean[df_clean['Patient_ID'] == patient_id]
                
            if not row.empty:
                # Konversi label target ke biner (1 atau 0)
                raw_label = str(row[target_label].values[0]).lower()
                label_val = 1 if 'appendicitis' in raw_label and 'no ' not in raw_label else 0
                
                # Ambil fitur dan konsep
                features = row[TABULAR_FEATURES].values[0].tolist()
                concepts = row[CONCEPTS].values[0].tolist()
                
                mapping[filename] = [filename, int(label_val), features, concepts]
        except Exception as e:
            continue
            
    os.makedirs(os.path.dirname(output_filepath), exist_ok=True)
    with open(output_filepath, 'w') as f:
        json.dump(mapping, f)
        
    print(f"Selesai. {len(mapping)} gambar berhasil dimapping dan disimpan ke JSON.\n")

# 1. Generate JSON (Langsung buang NaN)
TRAIN_JSON = os.path.join(OUTPUT_DIR, TARGET_LABEL, 'app_data_train.json')
TEST_JSON = os.path.join(OUTPUT_DIR, TARGET_LABEL, 'app_data_test.json')

generate_json_mapping_dropna(TRAIN_PROCESSED, CLINICAL_DATA_FILE, TRAIN_JSON, 'Diagnosis')
generate_json_mapping_dropna(TEST_PROCESSED, CLINICAL_DATA_FILE, TEST_JSON, 'Diagnosis')

# 2. Cek hasil pembuatan JSON
with open(TRAIN_JSON) as f:
    train_data = json.load(f)
with open(TEST_JSON) as f:
    test_data = json.load(f)

print(f"Total gambar di Train JSON: {len(train_data)}")
print(f"Total gambar di Test JSON: {len(test_data)}")

print("\n✅ SEMUA PROSES PREPROCESSING SELESAI TANPA IMPUTASI!")
print(f"File JSON siap pakai ada di: {os.path.join(OUTPUT_DIR, TARGET_LABEL)}")

Membuat mapping JSON untuk cropped_train (Mode: Drop NaN)...
Jumlah data awal di Excel: 782 baris
Jumlah data setelah drop NaN: 707 baris
Selesai. 1291 gambar berhasil dimapping dan disimpan ke JSON.

Membuat mapping JSON untuk cropped_test (Mode: Drop NaN)...
Jumlah data awal di Excel: 782 baris
Jumlah data setelah drop NaN: 707 baris
Selesai. 182 gambar berhasil dimapping dan disimpan ke JSON.

Total gambar di Train JSON: 1291
Total gambar di Test JSON: 182

✅ SEMUA PROSES PREPROCESSING SELESAI TANPA IMPUTASI!
File JSON siap pakai ada di: /kaggle/working/Processed/Dictionaries/diagnosis


# Training and Evaluation

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import models, transforms
from PIL import Image
import json
import os
import numpy as np
from sklearn.metrics import accuracy_score

# ==========================================
# 1. KONFIGURASI (Sesuaikan Path-nya)
# ==========================================
TRAIN_IMG_DIR = '/kaggle/working/Processed/Preprocessed/cropped_train'
TEST_IMG_DIR = '/kaggle/working/Processed/Preprocessed/cropped_test'
TRAIN_JSON = '/kaggle/working/Processed/Dictionaries/diagnosis/app_data_train.json'
TEST_JSON = '/kaggle/working/Processed/Dictionaries/diagnosis/app_data_test.json'

BATCH_SIZE = 8
EPOCHS = 1  # Hanya 1 epoch untuk testing
LEARNING_RATE = 1e-3
DEVICE = torch.device("cpu") # Paksa pakai CPU

# ==========================================
# 2. CUSTOM DATASET
# ==========================================
class AppendicitisDataset(Dataset):
    def __init__(self, json_file, img_dir, transform=None):
        with open(json_file, 'r') as f:
            self.mapping = json.load(f)
        self.img_dir = img_dir
        self.keys = list(self.mapping.keys())
        self.transform = transform

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, idx):
        img_name = self.keys[idx]
        data = self.mapping[img_name]
        label = data[1]
        features = data[2]
        
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
            
        return image, torch.tensor(features, dtype=torch.float32), torch.tensor(label, dtype=torch.float32)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load Data Asli
train_dataset_full = AppendicitisDataset(TRAIN_JSON, TRAIN_IMG_DIR, transform=transform)
test_dataset_full = AppendicitisDataset(TEST_JSON, TEST_IMG_DIR, transform=transform)

# BATASI DATA: Ambil maksimal 32 gambar train dan 16 gambar test
train_indices = list(range(min(32, len(train_dataset_full))))
test_indices = list(range(min(16, len(test_dataset_full))))

train_dataset = Subset(train_dataset_full, train_indices)
test_dataset = Subset(test_dataset_full, test_indices)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# ==========================================
# 3. MODEL ARSITEKTUR (Tanpa Pretrained agar Cepat)
# ==========================================
class SimpleMultimodalCNN(nn.Module):
    def __init__(self, num_tabular_features=6):
        super(SimpleMultimodalCNN, self).__init__()
        
        # Pretrained=False agar tidak perlu download bobot (instan di CPU)
        self.cnn = models.resnet18(pretrained=False) 
        num_ftrs = self.cnn.fc.in_features
        self.cnn.fc = nn.Identity() 
        
        self.tabular_mlp = nn.Sequential(
            nn.Linear(num_tabular_features, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU()
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(num_ftrs + 32, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, image, tabular_features):
        img_features = self.cnn(image)
        tab_features = self.tabular_mlp(tabular_features)
        combined = torch.cat((img_features, tab_features), dim=1)
        output = self.classifier(combined)
        return output.squeeze(1)

model = SimpleMultimodalCNN(num_tabular_features=6).to(DEVICE)

# ==========================================
# 4. TRAINING LOOP
# ==========================================
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("Memulai Training (Fast Run di CPU)...")
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    
    for batch_idx, (images, tabular, labels) in enumerate(train_loader):
        images, tabular, labels = images.to(DEVICE), tabular.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(images, tabular)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        print(f"  Batch {batch_idx+1}/{len(train_loader)} | Loss: {loss.item():.4f}")
        
    print(f"Epoch {epoch+1}/{EPOCHS} Selesai | Avg Train Loss: {running_loss/len(train_loader):.4f}")

# ==========================================
# 5. EVALUASI AKHIR
# ==========================================
print("\n--- Evaluasi pada Test Set ---")
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, tabular, labels in test_loader:
        images, tabular, labels = images.to(DEVICE), tabular.to(DEVICE), labels.to(DEVICE)
        
        outputs = model(images, tabular)
        preds = (torch.sigmoid(outputs) >= 0.5).float()
        
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

print(f"Total data test yang dievaluasi: {len(all_labels)}")
print(f"Label Asli: {all_labels}")
print(f"Label Prediksi: {all_preds}")

# Hitung akurasi jika ada lebih dari 0 data
if len(all_labels) > 0:
    acc = accuracy_score(all_labels, all_preds)
    print(f"Accuracy (Acak karena 1 epoch): {acc:.4f}")
else:
    print("Tidak ada data test yang dievaluasi.")

print("\n✅ KODE BERFUNGSI DENGAN BAIK!")

Memulai Training (Fast Run di CPU)...
  Batch 1/4 | Loss: 0.9040
  Batch 2/4 | Loss: 0.6553
  Batch 3/4 | Loss: 0.5992
  Batch 4/4 | Loss: 0.5170
Epoch 1/1 Selesai | Avg Train Loss: 0.6689

--- Evaluasi pada Test Set ---
Total data test yang dievaluasi: 16
Label Asli: [np.float32(0.0), np.float32(1.0), np.float32(1.0), np.float32(1.0), np.float32(0.0), np.float32(0.0), np.float32(1.0), np.float32(1.0), np.float32(1.0), np.float32(1.0), np.float32(1.0), np.float32(1.0), np.float32(1.0), np.float32(1.0), np.float32(0.0), np.float32(0.0)]
Label Prediksi: [np.float32(0.0), np.float32(0.0), np.float32(0.0), np.float32(0.0), np.float32(0.0), np.float32(0.0), np.float32(0.0), np.float32(0.0), np.float32(0.0), np.float32(0.0), np.float32(0.0), np.float32(0.0), np.float32(0.0), np.float32(0.0), np.float32(0.0), np.float32(0.0)]
Accuracy (Acak karena 1 epoch): 0.3125

✅ KODE BERFUNGSI DENGAN BAIK!
